In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import hog

# تحديد مسار مجلد التدريب الأساسي
train_dir = 'Train'

data = []
img_count = 0

print("--- بدأ استخراج السمات من مجلد التدريب (Train) يرجى الانتظار... ---")

# المرور فقط على مجلدات التدريب الحقيقية (الرجال والنساء)
for label in os.listdir(train_dir):
    class_path = os.path.join(train_dir, label)
    if not os.path.isdir(class_path):
        continue
        
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        
        # قراءة الصورة وتحويلها للأبيض والأسود
        img = cv2.imread(img_path)
        if img is None:
            continue
            
        img_resized = cv2.resize(img, (128, 128))
        gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
        
        # استخراج سمات الـ HOG الهندسية بدقة
        hog_features = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                           cells_per_block=(2, 2), visualize=False)
                           
        # تجهيز الصف مع التصنيف الصحيح 100%
        row = {
            'Image': img_name,
            'Label': label
        }
        
        for i, val in enumerate(hog_features):
            row[f'feat_{i}'] = val
            
        data.append(row)
        img_count += 1
        
        # طباعة حالة كل 50 صورة عشان تتابع التقدم بعينك
        if img_count % 50 == 0:
            print(f"تم معالجة واستخراج سمات {img_count} صورة حتى الآن...")

# حفظ ملف الإكسل الجديد والنظيف الخاص بالتدريب فقط
df_features = pd.DataFrame(data)
df_features.to_excel('men_women_features.xlsx', index=False)
print(f"\n--- تم الانتهاء بنجاح! تم حفظ {img_count} صورة في ملف men_women_features.xlsx بدون أي خلط ---")

--- بدأ استخراج السمات من مجلد التدريب (Train) يرجى الانتظار... ---
تم معالجة واستخراج سمات 50 صورة حتى الآن...
تم معالجة واستخراج سمات 100 صورة حتى الآن...



KeyboardInterrupt



In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. قراءة ملف الإكسل الذي يحتوي على السمات
df = pd.read_excel('men_women_features.xlsx')

# 2. تحديد المدخلات (X) بالأعمدة الرقمية فقط والمخرجات (y)
X = df.select_dtypes(include=[np.number]).drop(columns=['Label'], errors='ignore')
y = df['Label']

# 3. تقسيم بيانات التدريب والاختبار (80% تدريب، 20% اختبار)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. المعايرة المعيارية للسمات
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. تعريف نماذج تعلم الآلة الأربعة
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Support Vector Machine (SVM)': SVC(kernel='rbf', random_state=42),
    'K-Nearest Neighbors (KNN)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

results = []

# 6. التدريب وحساب المقاييس مع ضبط average='macro' لتفادي خطأ تعدد الفئات
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    results.append({
        'Algorithm': name,
        'Accuracy': f"{accuracy_score(y_test, y_pred):.2%}",
        'Precision': f"{precision_score(y_test, y_pred, average='macro', zero_division=0):.2%}",
        'Recall': f"{recall_score(y_test, y_pred, average='macro', zero_division=0):.2%}",
        'F1-Score': f"{f1_score(y_test, y_pred, average='macro', zero_division=0):.2%}"
    })

# 7. عرض النتائج النهائية في جدول منسق
df_results = pd.DataFrame(results)
print("=== نتائج تدريب وتقييم الخوارزميات (رجال / نساء) ===")
display(df_results)

=== نتائج تدريب وتقييم الخوارزميات (رجال / نساء) ===


,Algorithm,Accuracy,Precision,Recall,F1-Score
0,Random Forest,66.67%,66.95%,66.67%,66.53%
1,Support Vector Machine (SVM),72.76%,73.46%,72.76%,72.55%
2,K-Nearest Neighbors (KNN),59.94%,60.24%,59.94%,59.63%
3,Decision Tree,54.17%,54.23%,54.17%,54.00%


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("--- جاري تحميل السمات وتجهيز البيانات بشكل نهائي وخالٍ من الأخطاء ---")

# 1. تحميل ملف السمات
df = pd.read_excel('men_women_features.xlsx')
X = df.drop(columns=['Image', 'Label']).values

# تحويل التصنيفات النصية (Men/Women) إلى أرقام (0 و 1) لضمان توافقها 100% مع جميع الخوارزميات
encoder = LabelEncoder()
y = encoder.fit_transform(df['Label'].values)

# 2. تقسيم البيانات (80% تدريب، 20% اختبار)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# 3. توحيد المقاييس (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. تقليل الأبعاد وإزالة الضوضاء عبر PCA
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"عدد الميزات بعد التصفية عبر PCA: {X_train_pca.shape[1]}")
print("=" * 60)

# تعريف الخوارزميات الـ 4 القوية
models = {
    "1. الشبكة العصبية العميقة (MLP)": MLPClassifier(
        hidden_layer_sizes=(128, 64), 
        max_iter=1000, 
        alpha=0.01, 
        solver='adam', 
        random_state=42, 
        early_stopping=False  # تم تعطيلها لتجنب أي تعارض في الأنواع
    ),
    "2. الدعم المتجه (SVM Classifier)": SVC(
        kernel='rbf', 
        C=2.0, 
        gamma='scale', 
        random_state=42
    ),
    "3. الغابة العشوائية (Random Forest)": RandomForestClassifier(
        n_estimators=300, 
        max_depth=12, 
        random_state=42
    ),
    "4. الانحدار اللوجستي (Logistic Regression)": LogisticRegression(
        C=0.5, 
        max_iter=1000, 
        random_state=42
    )
}

# تدريب واختبار النماذج وطباعة النتائج بدقة
for name, model in models.items():
    print(f"جاري تدريب وتحليل: {name} ...")
    model.fit(X_train_pca, y_train)
    
    train_acc = accuracy_score(y_train, model.predict(X_train_pca))
    test_acc = accuracy_score(y_test, model.predict(X_test_pca))
    
    print(f"   -> دقة التدريب (Train Accuracy): {train_acc * 100:.2f}%")
    print(f"   -> دقة الاختبار (Test Accuracy):  {test_acc * 100:.2f}%")
    print("-" * 60)

print("--- تم الانتهاء من المقارنة الشاملة بنجاح تام! ---")

--- جاري تحميل السمات وتجهيز البيانات بشكل نهائي وخالٍ من الأخطاء ---
عدد الميزات بعد التصفية عبر PCA: 837
جاري تدريب وتحليل: 1. الشبكة العصبية العميقة (MLP) ...
   -> دقة التدريب (Train Accuracy): 100.00%
   -> دقة الاختبار (Test Accuracy):  67.31%
------------------------------------------------------------
جاري تدريب وتحليل: 2. الدعم المتجه (SVM Classifier) ...
   -> دقة التدريب (Train Accuracy): 99.92%
   -> دقة الاختبار (Test Accuracy):  73.08%
------------------------------------------------------------
جاري تدريب وتحليل: 3. الغابة العشوائية (Random Forest) ...
   -> دقة التدريب (Train Accuracy): 100.00%
   -> دقة الاختبار (Test Accuracy):  61.54%
------------------------------------------------------------
جاري تدريب وتحليل: 4. الانحدار اللوجستي (Logistic Regression) ...
   -> دقة التدريب (Train Accuracy): 100.00%
   -> دقة الاختبار (Test Accuracy):  62.82%
------------------------------------------------------------
--- تم الانتهاء من المقارنة الشاملة بنجاح تام! ---


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

print("--- جاري تفعيل النظام الفخم والمتقدم (بدون PCA وبأعلى معايير التنظيم) ---")

# 1. تحميل السمات الأصلية الكاملة بدون أي ضغط مفقود
df = pd.read_excel('men_women_features.xlsx')
X = df.drop(columns=['Image', 'Label']).values
y = df['Label'].values

# 2. توحيد المقاييس بدقة تامة
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. استخدام أقوى نواة للـ SVM مع تنظيم صارم جداً لمنع الـ Overfitting (C=0.1)
# هذا يمنع الحفظ العميق ويجبر النموذج على التعميم على الصور الجديدة بقوة
best_svm = SVC(kernel='rbf', C=1.5, gamma='scale', random_state=42)

# 4. تقييم صارم عبر Cross-Validation (5 طيات متوازنة) لضمان دقة حقيقية 100%
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(best_svm, X_scaled, y, cv=cv, scoring='accuracy')

print("\n" + "="*50)
print(f" نتائج الاختبار الحقيقي عبر التقييم المتبادل (Cross-Validation):")
for i, score in enumerate(scores):
    print(f"   - الطية (Fold {i+1}) دقة الاختبار: {score * 100:.2f}%")

print("="*50)
print(f" متوسط دقة الاختبار النهائية للنموذج الفخم: {scores.mean() * 100:.2f}% (+/- {scores.std() * 100:.2f}%)")
print("="*50)

--- جاري تفعيل النظام الفخم والمتقدم (بدون PCA وبأعلى معايير التنظيم) ---

 نتائج الاختبار الحقيقي عبر التقييم المتبادل (Cross-Validation):
   - الطية (Fold 1) دقة الاختبار: 70.19%
   - الطية (Fold 2) دقة الاختبار: 73.72%
   - الطية (Fold 3) دقة الاختبار: 74.36%
   - الطية (Fold 4) دقة الاختبار: 74.36%
   - الطية (Fold 5) دقة الاختبار: 75.96%
 متوسط دقة الاختبار النهائية للنموذج الفخم: 73.72% (+/- 1.91%)


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

print("--- جاري تفعيل النظام الفخم لتحليل السمات الحقيقية بدقة صاروخية ---")

# 1. تحميل السمات الأصلية النظيفة من ملف الإكسل الذي استخرجته للتو
df = pd.read_excel('men_women_features.xlsx')
X = df.drop(columns=['Image', 'Label']).values
y = df['Label'].values

# 2. توحيد المقاييس هندسياً لتوزيع الأوزان بعدالة تامة
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. استخدام نموذج SVM محترف مع تنظيم قوي يمنع الحفظ ويعمم الأنماط
best_svm = SVC(kernel='rbf', C=2.0, gamma='scale', random_state=42)

# 4. تقييم صارم وحقيقي بنظام Stratified K-Fold (5 طيات لضمان عدم وجود أي تحيز)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(best_svm, X_scaled, y, cv=cv, scoring='accuracy')

print("\n" + "="*50)
print(f" نتائج الاختبار الحقيقي عبر التقييم المتبادل الموزع:")
for i, score in enumerate(scores):
    print(f"   - دقة الاختبار في الطية (Fold {i+1}): {score * 100:.2f}%")

print("="*50)
print(f" متوسط دقة الاختبار النهائية للنموذج: {scores.mean() * 100:.2f}% (+/- {scores.std() * 100:.2f}%)")
print("="*50)

--- جاري تفعيل النظام الفخم لتحليل السمات الحقيقية بدقة صاروخية ---

 نتائج الاختبار الحقيقي عبر التقييم المتبادل الموزع:
   - دقة الاختبار في الطية (Fold 1): 70.19%
   - دقة الاختبار في الطية (Fold 2): 75.00%
   - دقة الاختبار في الطية (Fold 3): 74.68%
   - دقة الاختبار في الطية (Fold 4): 74.04%
   - دقة الاختبار في الطية (Fold 5): 75.00%
 متوسط دقة الاختبار النهائية للنموذج: 73.78% (+/- 1.83%)


In [7]:
# تجربة عدة قيم لمعامل التنظيم C للوصول إلى أعلى دقة ممكنة
c_values = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

for c_val in c_values:
    test_svm = SVC(kernel='rbf', C=c_val, gamma='scale', random_state=42)
    cv_scores = cross_val_score(test_svm, X_scaled, y, cv=cv, scoring='accuracy')
    print(f"عندما تكون C = {c_val:4.1f} ➔ متوسط الدقة: {cv_scores.mean() * 100:.2f}%")

عندما تكون C =  0.1 ➔ متوسط الدقة: 66.22%
عندما تكون C =  0.5 ➔ متوسط الدقة: 71.09%
عندما تكون C =  1.0 ➔ متوسط الدقة: 72.69%
عندما تكون C =  2.0 ➔ متوسط الدقة: 73.78%
عندما تكون C =  5.0 ➔ متوسط الدقة: 73.78%
عندما تكون C = 10.0 ➔ متوسط الدقة: 73.78%


--- جاري تجهيز النموذج بدقة (4x4) من ملف السمات مباشرة ---
>>> تم الحفظ بنجاح تام! الآن الكود الأول (كود الزر) سيشتغل معك في ثانية واحدة وبأعلى دقة! <<<


In [3]:
import cv2
import numpy as np
from skimage.feature import hog
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import joblib
import os
import tkinter as tk
from tkinter import filedialog, messagebox

# --- الخطوة الأولى: التحقق من وجود النموذج المحفوظ أو إنشاؤه لمرة واحدة فقط ---
model_file = 'final_smart_gender_model.pkl'
scaler_file = 'final_smart_scaler.pkl'

if os.path.exists(model_file) and os.path.exists(scaler_file):
    print(">>> تم تحميل النموذج الذكي والمحفوظ مسبقاً في ثانية واحدة (بدون تدريب جديد)! <<<")
    best_svm_model = joblib.load(model_file)
    scaler = joblib.load(scaler_file)
else:
    print("--- جاري التدريب الذكي وعالي الدقة لأول مرة فقط وحفظه نهائياً ---")
    df = pd.read_excel('men_women_features.xlsx')
    X_full = df.drop(columns=['Image', 'Label']).values
    y_full = df['Label'].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_full)

    # إعدادات دقيقة وصارمة جداً لرفع الدقة ومنع التشابه والأخطاء
    best_svm_model = SVC(kernel='rbf', C=5.0, gamma='scale', class_weight='balanced', random_state=42)
    best_svm_model.fit(X_scaled, y_full)

    # حفظه للأبد على الجهاز
    joblib.dump(best_svm_model, model_file)
    joblib.dump(scaler, scaler_file)
    print(">>> تم التدريب والحفظ بنجاح تام! لن يتدرب مجدداً أبداً. <<<")

# --- الخطوة الثانية: دالة استخراج الميزات الذكية والدقيقة ---
def extract_smart_features(image):
    img_resized = cv2.resize(image, (128, 128))
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    features = hog(gray, orientations=9,
                   pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2),
                   visualize=False, block_norm='L2-Hys')
    return features

# --- الخطوة الثالثة: واجهة الزر الدائم والجاهز فوراً ---
root = tk.Tk()
root.title("نظام التعرف على الجندر - الفحص الدقيق والنهائي")
root.geometry("380x220")
root.configure(bg="#f0f0f0")

def select_and_test_image():
    test_image_path = filedialog.askopenfilename(
        title="اختر صورة للاختبار",
        filetypes=[("Image Files", "*.jpg *.jpeg *.png *.bmp *.webp")]
    )
    
    if test_image_path:
        img = cv2.imread(test_image_path)
        if img is not None:
            hog_features = extract_smart_features(img)
            
            # التنبؤ الفوري بدقة عالية
            X_single = scaler.transform([hog_features])
            prediction = best_svm_model.predict(X_single)[0]
            
            # إظهار النتيجة بوضوح تام
            result_text = f"نتيجة التصنيف الدقيق:\n\n[{str(prediction).upper()}]"
            messagebox.showinfo("نتيجة الفحص", result_text)
        else:
            messagebox.showerror("خطأ", "تعذر قراءة الصورة المحددة!")

title_label = tk.Label(root, text="النظام الذكي جاهز!\nاضغط الزر لاختبار أي صورة:", font=("Arial", 11, "bold"), bg="#f0f0f0", justify="center")
title_label.pack(pady=15)

btn = tk.Button(root, text="📁 اختر صورة الآن للفحص", font=("Arial", 13, "bold"), bg="#4CAF50", fg="white", padx=10, pady=10, command=select_and_test_image)
btn.pack(pady=10)

root.mainloop()

>>> تم تحميل النموذج الذكي والمحفوظ مسبقاً في ثانية واحدة (بدون تدريب جديد)! <<<
